In [1]:
%pip install pandas numpy scikit-learn tensorflow joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
data = pd.read_csv("combined_dataset.csv")

print("First 5 Rows:")
print(data.head())
print("\nDataset Shape:")
print(data.shape)
print("\nColumns:")
print(data.columns)
print("\nMissing Values:")
print(data.isnull().sum())

First 5 Rows:
  target                                               text
0   spam  Congratulations! You've been selected for a lu...
1   spam  URGENT: Your account has been compromised. Cli...
2   spam  You've won a free iPhone! Claim your prize by ...
3   spam  Act now and receive a 50% discount on all purc...
4   spam  Important notice: Your subscription will expir...

Dataset Shape:
(10961, 2)

Columns:
Index(['target', 'text'], dtype='object')

Missing Values:
target    0
text      0
dtype: int64


In [4]:
data = data[["text", "target"]]

data["target"] = data["target"].map({
    "ham": 0,
    "spam": 1
})

data.dropna(inplace=True)

data = data.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(data.head())
print("\nClass Distribution:")
print(data["target"].value_counts())

                                                text  target
0  enw meeting / teambuilding event - jillian ' s...       0
1  organizational announcement please see attache...       0
2  Is it your yahoo boys that bring in the perf? ...       0
3  Get 3 Lions England tone, reply lionm 4 mono o...       1
4  Yup... I havent been there before... You want ...       0

Class Distribution:
target
0    8555
1    2406
Name: count, dtype: int64


In [5]:
X = data["text"]
y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Samples :", len(X_train))
print("Testing Samples :", len(X_test))

Training Samples : 8768
Testing Samples : 2193


In [6]:
vocab_size = 10000
max_length = 100

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_test_sequences = tokenizer.texts_to_sequences(X_test)

In [7]:
X_train_padded = pad_sequences(
    X_train_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_test_padded = pad_sequences(
    X_test_sequences,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

print(X_train_padded.shape)
print(X_test_padded.shape)

(8768, 100)
(2193, 100)


In [8]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = {
    0: weights[0],
    1: weights[1]
}

print(class_weights)

{0: np.float64(0.640654683618296), 1: np.float64(2.2774025974025975)}


In [9]:
model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_length
    )
)

model.add(
    LSTM(128)
)

model.add(
    Dropout(0.5)
)

model.add(
    Dense(
        64,
        activation="relu"
    )
)

model.add(
    Dense(
        1,
        activation="sigmoid"
    )
)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

c:\Users\Roshan\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_padded,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 33s 111ms/step - accuracy: 0.7706 - loss: 0.5802 - val_accuracy: 0.8198 - val_loss: 0.4571
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 23s 103ms/step - accuracy: 0.8296 - loss: 0.4553 - val_accuracy: 0.8101 - val_loss: 0.4157
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 24s 109ms/step - accuracy: 0.8005 - loss: 0.5416 - val_accuracy: 0.8301 - val_loss: 0.6337
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 23s 106ms/step - accuracy: 0.8256 - loss: 0.5440 - val_accuracy: 0.8352 - val_loss: 0.5602


In [11]:
loss, accuracy = model.evaluate(
    X_test_padded,
    y_test
)
print("\nTest Accuracy :", accuracy)

predictions = model.predict(X_test_padded)
predicted_labels = []

for prediction in predictions:

    if prediction >= 0.5:
        predicted_labels.append(1)
    else:
        predicted_labels.append(0)

print("\nAccuracy Score:")
print(accuracy_score(
    y_test,
    predicted_labels
))

print("\nClassification Report:")
print(classification_report(
    y_test,
    predicted_labels
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test,
    predicted_labels
))

69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - accuracy: 0.8171 - loss: 0.4068

Test Accuracy : 0.8171454668045044
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step

Accuracy Score:
0.8171454628362973

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.83      0.88      1712
           1       0.56      0.77      0.65       481

    accuracy                           0.82      2193
   macro avg       0.74      0.80      0.76      2193
weighted avg       0.85      0.82      0.83      2193


Confusion Matrix:
[[1420  292]
 [ 109  372]]


In [12]:
import joblib

# Save the trained model
model.save("spam_model.keras")

# Save the tokenizer
joblib.dump(tokenizer, "tokenizer.pkl")

print("Model and Tokenizer saved successfully!")

Model and Tokenizer saved successfully!


In [13]:
import os

print(os.getcwd())
print(os.listdir())

c:\Studies\internships\LivetechIndia\sms detect dataset
['app.py', 'bg.jpg', 'combined_dataset.csv', 'sms-frontend.html', 'spam detecting using lstm.ipynb', 'spam_model.keras', 'style.css', 'tokenizer.pkl']


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

while True:

    sms = input("Enter SMS (type 'exit' to quit): ")

    if sms.lower() == "exit":
        break

    # Convert text to sequence
    sequence = tokenizer.texts_to_sequences([sms])

    # Pad sequence (use the SAME max_length as training)
    padded = pad_sequences(
        sequence,
        maxlen=max_length, 
        padding="post"
    )

    # Predict
    prediction = model.predict(padded, verbose=0)[0][0]
    prediction = float(prediction)

    print("\nRaw Prediction:", prediction)

    if prediction >= 0.5:
        print("Prediction: SPAM")
        print("Confidence:", round(prediction * 100, 2), "%")
    else:
        print("Prediction: HAM")
        print("Confidence:", round((1 - prediction) * 100, 2), "%")

    print("-" * 40)